[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/bases-de-datos-vectoriales/02-chromadb-weaviate.ipynb)

# ChromaDB en Profundidad y Comparativa con Weaviate

> Domina ChromaDB para búsqueda vectorial local y comprende cuándo escalar a
> soluciones como Weaviate o Pinecone en producción.

## Objetivos
- Usar ChromaDB con colecciones, filtros de metadatos y búsqueda híbrida
- Implementar búsqueda híbrida (semántica + keywords)
- Construir un sistema de QA con Claude y ChromaDB
- Comparar ChromaDB, Weaviate y Pinecone: cuándo usar cada uno

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic chromadb sentence-transformers --quiet

## 2. ChromaDB: colecciones y metadatos

In [ ]:
import anthropic
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("Cargando modelo de embeddings...")
modelo_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
chroma = chromadb.EphemeralClient()

# Corpus con metadatos ricos
ARTICULOS = [
    {"id": "a1", "texto": "Python es el lenguaje más usado en data science gracias a NumPy y Pandas.", "categoria": "programacion", "nivel": "basico"},
    {"id": "a2", "texto": "Los transformers procesan secuencias en paralelo usando atención multi-cabezal.", "categoria": "ia", "nivel": "avanzado"},
    {"id": "a3", "texto": "Docker permite empaquetar aplicaciones con todas sus dependencias en contenedores.", "categoria": "devops", "nivel": "intermedio"},
    {"id": "a4", "texto": "El fine-tuning adapta modelos LLM a dominios específicos con menos datos.", "categoria": "ia", "nivel": "avanzado"},
    {"id": "a5", "texto": "FastAPI es un framework web moderno y rápido para construir APIs con Python.", "categoria": "programacion", "nivel": "intermedio"},
    {"id": "a6", "texto": "Kubernetes orquesta contenedores Docker a escala en clústeres de producción.", "categoria": "devops", "nivel": "avanzado"},
    {"id": "a7", "texto": "RAG mejora las respuestas de los LLMs recuperando información actualizada.", "categoria": "ia", "nivel": "intermedio"},
    {"id": "a8", "texto": "Git es el sistema de control de versiones distribuido más usado del mundo.", "categoria": "programacion", "nivel": "basico"},
    {"id": "a9", "texto": "Los embeddings multilingües capturan semántica en múltiples idiomas en el mismo espacio vectorial.", "categoria": "ia", "nivel": "avanzado"},
    {"id": "a10", "texto": "GitHub Actions automatiza pipelines de CI/CD directamente desde el repositorio.", "categoria": "devops", "nivel": "intermedio"},
]

# Crear colección con embeddings precomputados
coleccion = chroma.create_collection("articulos_tech")
textos = [a["texto"] for a in ARTICULOS]
embeddings = modelo_emb.encode(textos).tolist()

coleccion.add(
    ids=[a["id"] for a in ARTICULOS],
    documents=textos,
    embeddings=embeddings,
    metadatas=[{"categoria": a["categoria"], "nivel": a["nivel"]} for a in ARTICULOS]
)
print(f"✓ {len(ARTICULOS)} artículos indexados con metadatos.")

## 3. Búsqueda con filtros de metadatos

In [ ]:
def buscar_con_filtro(query: str, filtro: dict = None, n: int = 3) -> list:
    """Búsqueda semántica con filtros de metadatos opcionales."""
    query_emb = modelo_emb.encode([query]).tolist()
    kwargs = {"query_embeddings": query_emb, "n_results": n}
    if filtro:
        kwargs["where"] = filtro
    resultado = coleccion.query(**kwargs)
    return list(zip(resultado["documents"][0], resultado["metadatas"][0], resultado["distances"][0]))

# Búsqueda 1: sin filtro
print("=== SIN FILTRO ===")
for doc, meta, dist in buscar_con_filtro("¿Cómo desplegar aplicaciones a escala?"):
    print(f"  [{meta['categoria']}|{meta['nivel']}] (dist={dist:.3f}) {doc[:80]}")

# Búsqueda 2: solo artículos de IA
print("\n=== SOLO CATEGORÍA 'ia' ===")
for doc, meta, dist in buscar_con_filtro("modelos de lenguaje", filtro={"categoria": "ia"}):
    print(f"  [{meta['nivel']}] (dist={dist:.3f}) {doc[:80]}")

# Búsqueda 3: nivel avanzado de devops
print("\n=== DEVOPS NIVEL AVANZADO ===")
for doc, meta, dist in buscar_con_filtro("orquestación", filtro={"$and": [{"categoria": "devops"}, {"nivel": "avanzado"}]}):
    print(f"  (dist={dist:.3f}) {doc[:80]}")

## 4. Búsqueda híbrida: semántica + keywords

In [ ]:
def busqueda_hibrida(query: str, n: int = 3, alpha: float = 0.7) -> list:
    """
    Búsqueda híbrida combinando score semántico y BM25 simplificado.
    alpha: peso del score semántico (0=solo keywords, 1=solo semántico)
    """
    # Score semántico (ChromaDB devuelve distancias, convertir a similitudes)
    query_emb = modelo_emb.encode([query]).tolist()
    res = coleccion.query(query_embeddings=query_emb, n_results=len(ARTICULOS))
    scores_semantico = {doc: 1 - dist
                        for doc, dist in zip(res["documents"][0], res["distances"][0])}

    # Score BM25 simplificado (frecuencia de términos de la query)
    palabras_query = set(query.lower().split())
    scores_keyword = {}
    for art in ARTICULOS:
        palabras_doc = set(art["texto"].lower().split())
        coincidencias = len(palabras_query & palabras_doc)
        scores_keyword[art["texto"]] = coincidencias / max(len(palabras_query), 1)

    # Combinar scores
    scores_combinados = {}
    for doc in res["documents"][0]:
        sem = scores_semantico.get(doc, 0)
        kw = scores_keyword.get(doc, 0)
        kw_max = max(scores_keyword.values()) or 1
        scores_combinados[doc] = alpha * sem + (1 - alpha) * (kw / kw_max)

    top = sorted(scores_combinados.items(), key=lambda x: x[1], reverse=True)[:n]
    return top

QUERY_HIBRIDA = "Python framework API rápida"
print(f"Query: '{QUERY_HIBRIDA}'\n")
print("=== BÚSQUEDA SEMÁNTICA PURA (alpha=1.0) ===")
for doc, score in busqueda_hibrida(QUERY_HIBRIDA, alpha=1.0):
    print(f"  [{score:.3f}] {doc[:80]}")

print("\n=== BÚSQUEDA HÍBRIDA (alpha=0.7) ===")
for doc, score in busqueda_hibrida(QUERY_HIBRIDA, alpha=0.7):
    print(f"  [{score:.3f}] {doc[:80]}")

## 5. Sistema QA y comparativa de bases vectoriales

In [ ]:
import pandas as pd

def sistema_qa(pregunta: str) -> str:
    """Sistema QA con ChromaDB + Claude."""
    resultados = buscar_con_filtro(pregunta, n=3)
    contexto = "\n".join([f"- {doc[:120]}" for doc, _, _ in resultados])
    resp = client.messages.create(
        model=MODELO, max_tokens=300,
        messages=[{"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"}]
    ).content[0].text
    return resp

print("=== SISTEMA QA ===")
print(sistema_qa("¿Qué herramientas uso para desplegar contenedores a escala?"))

# Tabla comparativa
print("\n=== COMPARATIVA DE BASES DE DATOS VECTORIALES ===")
comparativa = pd.DataFrame([
    {"BD": "ChromaDB", "Hosting": "Local / Cloud", "Precio": "Open source", "Escala": "Millones", "Mejor para": "Prototipado y proyectos medianos"},
    {"BD": "Pinecone", "Hosting": "Cloud managed", "Precio": "Pago por uso", "Escala": "Miles de millones", "Mejor para": "Producción cloud sin ops"},
    {"BD": "Weaviate", "Hosting": "Local / Cloud", "Precio": "Open source / SaaS", "Escala": "Miles de millones", "Mejor para": "Búsqueda híbrida avanzada"},
    {"BD": "pgvector", "Hosting": "PostgreSQL", "Precio": "Open source", "Escala": "Millones", "Mejor para": "Integrar vectores en BD existente"},
    {"BD": "Qdrant", "Hosting": "Local / Cloud", "Precio": "Open source", "Escala": "Miles de millones", "Mejor para": "Alto rendimiento con filtros"},
])
print(comparativa.to_string(index=False))